## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/10/09 19:34:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.stop()

In [3]:
spark = None

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


spark

In [5]:
spark.sql("use hot.isk").show()
spark.sql("show tables").show()


++
||
++
++

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|      isk|transactions_old|      false|
|      isk|        accounts|      false|
|      isk|       customers|      false|
|      isk|        branches|      false|
|      isk|    transactions|      false|
+---------+----------------+-----------+



In [ ]:
spark.client.




### ISK view over Kafka has multiple topics including two topics with transactions data:
    
 - **transactions_old** with static set of data of 500000 events sorted by TransactionTime - that can be loaded into the cold set on MiniIO for demo purposes
 - **transactions** with dynamic set of data - starting with 50 events and gets new event every second.



In [ ]:
spark.sql("DROP TABLE cold.data.customers")

In [6]:
spark.sql("""
CREATE TABLE cold.data.customers 
USING iceberg 
TBLPROPERTIES('format-version'='2') 
PARTITIONED BY (kafka_partition, truncate(1000, kafka_offset)) 
AS 
  SELECT * 
  FROM hot.isk.customers;
""")

DataFrame[]

In [7]:
spark.sql("SELECT COUNT(*) FROM cold.data.customers").show();

+--------+
|count(1)|
+--------+
|  400000|
+--------+



In [22]:
spark.sql("DROP TABLE cold.data.index_customers_name")

DataFrame[]

In [23]:
spark.sql("""
CREATE TABLE cold.data.index_customers_name
USING iceberg 
TBLPROPERTIES('format-version'='2') 
PARTITIONED BY (bucket(1000,Name)) 
AS
  SELECT DISTINCT
    c.Name,
    c.kafka_partition,
    floor(c.kafka_offset/1000) offset_batch
  FROM
    cold.data.customers c
""")

DataFrame[]

In [9]:
import datetime

In [24]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers WHERE Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

2025-10-09 19:49:10.217070
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|CustomerID|        Name|             Address|               Email|       PhoneNo|kafka_partition|kafka_offset|            kafka_ts|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|    174178|Brendan Yost|Apt. 178 839 Bret...|su.wilderman@hotm...|(305) 225-5341|              0|        1000|2025-10-09 19:30:...|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+

2025-10-09 19:49:11.372648


In [32]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
WHERE c.Name = 'Brendan Yost' and c.kafka_offset >=0 and c.kafka_offset <=1000
""").show()
print( datetime.datetime.now())

2025-10-09 19:54:15.053408
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|CustomerID|        Name|             Address|               Email|       PhoneNo|kafka_partition|kafka_offset|            kafka_ts|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|    174178|Brendan Yost|Apt. 178 839 Bret...|su.wilderman@hotm...|(305) 225-5341|              0|        1000|2025-10-09 19:30:...|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+

2025-10-09 19:54:15.336026


In [26]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN cold.data.index_customers_name i
ON c.Name = i.Name AND c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

2025-10-09 19:49:22.054555
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+
|CustomerID|        Name|             Address|               Email|       PhoneNo|kafka_partition|kafka_offset|            kafka_ts|        Name|kafka_partition|offset_batch|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+
|    174178|Brendan Yost|Apt. 178 839 Bret...|su.wilderman@hotm...|(305) 225-5341|              0|        1000|2025-10-09 19:30:...|Brendan Yost|              0|           1|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+

2025-10-09 19:49:23.658484


In [27]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN (SELECT * FROM cold.data.index_customers_name WHERE Name='Brendan Yost') i
ON c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

2025-10-09 19:49:57.143865
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+
|CustomerID|        Name|             Address|               Email|       PhoneNo|kafka_partition|kafka_offset|            kafka_ts|        Name|kafka_partition|offset_batch|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+
|    174178|Brendan Yost|Apt. 178 839 Bret...|su.wilderman@hotm...|(305) 225-5341|              0|        1000|2025-10-09 19:30:...|Brendan Yost|              0|           1|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+------------+---------------+------------+

2025-10-09 19:49:58.908496
